# N112 · 状态压缩DP与集合状态

**目标：** 把已选择对象集合编码成充分状态。

**先修：** N098, N124, N101, N064。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** mask；最后位置；子集转移；访问状态图；状态数与转移数。

**配套讲解来源：** [同名逐章意见](../../comment/112_bitmask_dp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

有一类问题的"处境"不是单个数字，而是"**我已经收集了哪些东西**"这种集合：已经覆盖了哪些技能、已经拼出了哪些字母、已经访问了哪些城市。集合没法直接当下标，但当集合很小（20 来个元素以内）时，可以把它编码成一个整数——第 i 位是 1 就代表"元素 i 已收集"。这个整数叫掩码（mask），拿它当 DP 状态，就叫状态压缩 DP。三个代表问题：

1. **最小的必要团队**：每项技能至少要一个人会，求人数最少的团队。（LeetCode 1125）
2. **贴纸拼词**：每张贴纸可以撕一次、取走它的全部字母，问最少几张贴纸能凑出目标词；凑不出返回 −1。（LeetCode 691）
3. **哈密顿路径**：n 个城市两两之间有路费，每个城市恰好访问一次的一条路线叫哈密顿路径，求最小总路费。（对应经典的状压 DP 模型）

先看一个具体到数字的例子（“运行示例”部分用的就是它）：

- 输入技能 `['python','sql','js']`，三个人 `[['python'], ['sql'], ['sql','js']]`（0 号会 python，1 号只会 sql，2 号会 sql 和 js）。
- 输出 `[0, 2]`。
- 为什么是 [0,2]？三项技能分别编号 0、1、2。选 0 号（覆盖 python）加 2 号（覆盖 sql、js），三种技能全齐，只用了 2 人；任何 1 人的组合最多覆盖两种技能，所以 2 人就是最少。

## 2. 基础知识：先读懂这些词

- **mask（位掩码）**：一个整数拿来当集合用，第 i 个二进制位是 1 表示元素 i 在集合里。三个技能的"全收集"状态就是二进制 111，也就是 7；代码里 `(1<<len(skills))-1` 生成的就是这种"全 1"掩码。
- **状态压缩 DP**：把集合编码进一个整数的下标里做 DP。它并没有消灭指数复杂度——n 个元素的集合仍有 2ⁿ 种状态——只是把指数装进了一个整数里，让程序跑得动、写得清（推导段原话：压缩只是让它可控而非变成多项式）。
- **状态必须"充分"**：选状态的原则是"把会影响未来的全部信息装进去"。团队问题里未来只关心"已覆盖哪些技能"，所以掩码一个维度就够；旅行问题（哈密顿）里光知道"去过哪些城市"不够，下一步能去哪、花多少钱还取决于"现在站在哪个城市"，所以还要加一维"最后位置"。
- **最后位置（last position）**：哈密顿 DP 的第二维。`dp[mask][u]` = "访问过 mask 这些城市、当前停在城市 u"的最小费用。转移时往 mask 里追加一个没去过的城市 v，费用加上 `cost[u][v]`。
- **支配（dominance）**：两个方案若覆盖的技能集合相同，人数多的那个永远不比人数少的好，直接丢弃、只保留最省的。这是团队问题 dp 字典里那个长度比较的含义。
- **快照遍历（`list(dp.items())`）**：团队问题里每个人只处理一轮，处理时先把字典当时的内容拍个快照再遍历，防止"本轮新加入的方案"又被同一个人扩充一遍（那等于把一个人选了两次）。
- **记忆化 + 元组状态（`@cache`）**：贴纸问题把"每种字母还缺几个"打包成元组当递归参数，`@cache` 自动缓存（memoization：算过的答案存起来，下次直接查，不再重算）。
- **状态数与转移数**：这是两笔不同的账。哈密顿有 2ⁿ·n 个状态，但每个状态要试 n² 种"最后一条边"的接法，总转移量是 O(n²·2ⁿ)。分清这两笔账是练习 2 的主题。

## 3. 思路推导：从小例子开始

### 主线：最小团队，在 3 技能 3 人例子上手算

- **Step 1：技能编号。** `pos={'python':0,'sql':1,'js':2}`，全收集掩码 `full=0b111=7`。“运行示例”部分把每个人的掩码打成了表：

| 人 | 技能 | 掩码 |
|---|---|---|
| 0 | ['python'] | 001 |
| 1 | ['sql'] | 010 |
| 2 | ['sql','js'] | 110 |

- **Step 2：dp 字典。** 键 = 已覆盖的技能掩码，值 = 达成它的人员下标元组（保留最短的）。初始 `dp={0:()}`——什么都不覆盖，零人。
- **Step 3：逐人扩充（0/1 背包式的"每人一轮"）。** 对每个人，拿他处理前的快照逐项尝试"加进团队"：
  - 人 0（掩码 001）：从空集出发得到覆盖 001 的方案 `(0,)`。dp 现有 {0:(), 001:(0,)}。
  - 人 1（掩码 010）：从快照的两项分别扩展：000|010=010 → `(1,)`；001|010=011 → `(0,1)`。dp 现有四项。
  - 人 2（掩码 110）：从快照的四项扩展：000→110 得 `(2,)`；001→111 得 `(0,2)`；010→110 已存在且现有方案 `(2,)` 只要 1 人，2 人的新方案被支配丢弃；011→111 已存在 `(0,2)` 更短，丢弃。
- **Step 4：读答案。** `dp[full]=dp[7]=(0,2)`，返回 `[0,2]`。注意 Step 3 里两次"丢弃"就是支配在起作用，而快照保证了人 2 不会把自己用两遍。

### 副线一：贴纸 `'thehat'`、贴纸 `['with','example','science']`

- 目标词去重排序得字母表 `letters=['a','e','h','t']`，还缺的量写成向量 `state=(a:1, e:1, h:2, t:2)`。每张贴纸也按同一字母表向量化：'with'→(0,0,1,1)，'example'→(1,1,0,0)，'science'→(0,1,0,0)。
- 递归 `f(state)` = 还缺 state 时最少还要几张。剪枝关键：贴纸使用的**顺序无关**，所以可以规定"下一张贴纸必须覆盖第一个还没凑够的字母"。
- 手算：state=(1,1,2,2)，首个缺口是 a，能补 a 的只有 'example' → 用掉 1 张，state 变 (0,0,2,2)。首个缺口是 h，能补 h 的只有 'with' → state 变 (0,0,1,1)，再来一张 'with' → (0,0,0,0)，归零。共 **3** 张。
- 终止性：每用一张"必含首个缺口字母"的贴纸，那个坐标严格减小，最多递归有限层；所有贴纸都补不了首个缺口时（比如目标里有字母而贴纸全没有），best 保持无穷，最终返回 −1。

### 副线二：哈密顿路径，3 城市小例

- 路费表 `cost=[[0,1,10],[1,0,1],[10,1,0]]`（0↔1 花 1，1↔2 花 1，0↔2 直达要 10）。
- `dp[只含u][u]=0` 是起点（一个人出发）。转移：从"mask 城市、停在 u"追加未访问的 v，费用加 `cost[u][v]`。
- 推几个格子：dp[{0,1}][1] = dp[{0}][0]+cost[0][1] = 0+1 = 1（走 0→1）。dp[{1,2}][2] = dp[{1}][1]+1 = 1（走 1→2）。dp[全][2] = dp[{0,1}][1]+1 = 2（路线 0→1→2，绕开了 10 块的直达路）。
- 答案 = min(dp[全集][各终点]) = 2。这里就体现了"最后位置"维度的用处：同样访问了 {0,1}，停在 0 还是停在 1，下一步的价钱完全不同。

## 4. 核心代码：smallest_sufficient_team

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def smallest_sufficient_team(skills, people):
    pos = {s: i for i, s in enumerate(skills)}
    full = (1 << len(skills)) - 1
    dp = {0: ()}
    for i, person in enumerate(people):
        mask = 0
        for s in person:
            if s in pos:
                mask |= 1 << pos[s]
        for covered, team in list(dp.items()):
            new = covered | mask
            if new not in dp or len(team) + 1 < len(dp[new]):
                dp[new] = team + (i,)
    return list(dp[full]) if full in dp else None
```

### 逐段读懂代码

### 1. `smallest_sufficient_team`

```python
def smallest_sufficient_team(skills, people):
    pos = {s: i for i, s in enumerate(skills)}
    full = (1 << len(skills)) - 1
    dp = {0: ()}
    for i, person in enumerate(people):
        mask = 0
        for s in person:
            if s in pos:
                mask |= 1 << pos[s]
        for covered, team in list(dp.items()):
            new = covered | mask
            if new not in dp or len(team) + 1 < len(dp[new]):
                dp[new] = team + (i,)
    return list(dp[full]) if full in dp else None
```

- `pos` 给每项技能一个二进制位；`full` 是全 1 掩码；`dp={0:()}` 是初始状态（覆盖空集、零人）。
- 内两层 for 给第 i 个人算掩码：他会的每项技能把对应位点亮（`mask|=1<<pos[s]`）；`if s in pos` 挡住"这个人会的技能不在需求清单里"的干扰。
- `for covered,team in list(dp.items())`：先 `list(...)` 拍快照再遍历，本轮内新加入 dp 的方案不会被人 i 再扩展一遍——这就是"每人只处理一轮"。
- `new=covered|mask` 把两边的技能并起来；判断 `if new not in dp or len(team)+1<len(dp[new])` 是支配：新方案（人数 = 原团队 + 1）只有在"更省"时才值得记录。
- 收尾：`full` 可达就返回 `list(dp[full])`（元组转列表）；到不了（没人会某项必备技能）返回 None。

### 2. `min_stickers`

```python
def min_stickers(stickers, target):
    letters = sorted(set(target))
    wanted = Counter(target)
    vectors = [tuple((Counter(s)[c] for c in letters)) for s in stickers]

    @cache
    def f(state):
        if not any(state):
            return 0
        first = next((i for i, x in enumerate(state) if x))
        best = float('inf')
        for v in vectors:
            if not v[first]:
                continue
            new = tuple((max(0, x - y) for x, y in zip(state, v)))
            best = min(best, 1 + f(new))
        return best
    answer = f(tuple((wanted[c] for c in letters)))
    return -1 if answer == float('inf') else answer
```

- 第一行做向量化：`letters` 是目标词里出现过的字母（排序保证顺序稳定，状态才能一致）；每张贴纸变成"按 letters 顺序的各种字母数量"的元组（`Counter(s)[c]` 查贴纸 s 里字母 c 的个数，没有自然得 0）。
- `f(state)`：state 是"每种字母还缺几个"的元组。`if not any(state): return 0` 是递归基——全都缺 0 个，一张都不用再贴。
- `first=next(...)` 找到第一个还没凑够的字母下标，随后 `if not v[first]: continue` 只考虑能补这个字母的贴纸——这就是"贴纸顺序无关，强制处理首个缺口"的剪枝，它既砍掉了等价的重复顺序，又保证了那个坐标严格下降、递归必然终止。
- `new=tuple(max(0,x-y) for x,y in zip(state,v))`：每项缺口减去贴纸供给，`max(0,...)` 挡住减成负数（多出来的字母直接浪费）。`best=min(best,1+f(new))` 记录"用这张贴纸 + 之后最优"的最小值。
- `@cache` 让相同的 state 只算一次。最后把"无穷"翻译成 −1。

### 3. `hamiltonian_path_dp`

```python
def hamiltonian_path_dp(cost):
    n = len(cost)
    if not n:
        return 0
    if any((len(row) != n for row in cost)):
        raise ValueError('square cost matrix required')
    dp = [[float('inf')] * n for _ in range(1 << n)]
    for u in range(n):
        dp[1 << u][u] = 0
    for mask in range(1 << n):
        for u in range(n):
            if not mask >> u & 1:
                continue
            for v in range(n):
                if not mask >> v & 1:
                    dp[mask | 1 << v][v] = min(dp[mask | 1 << v][v], dp[mask][u] + cost[u][v])
    return min(dp[-1])
```

- 前三行是防御：空矩阵返回 0；`any(len(row)!=n ...)` 检查每一行长度都是 n（不是方阵直接报错，因为 `cost[u][v]` 会查错位）。
- `dp` 是 2ⁿ 行 n 列的表，全部初始化为正无穷（不可达）。`dp[1<<u][u]=0`：从城市 u 单独出发，费用 0——每个城市都可以是起点。
- 三重循环按 mask 从小到大填表（mask 变大意味着访问的城市变多，依赖的方向正确）：`mask>>u&1` 检查 u 是否在 mask 里（先把 mask 右移 u 位再取最低位），不在就 `continue`；对每个不在 mask 里的 v，走"追加 v"这条转移：新状态是 `mask|1<<v`（把 v 的位点亮），停在 v，费用取 `min` 与 `dp[mask][u]+cost[u][v]` 的较小者。
- `return min(dp[-1])`：`dp[-1]` 是 mask=2ⁿ−1（全访问）那一行，取各终点的最小值。如果图根本拼不出一条全访问路径，这行全是无穷，返回 +∞——调用方要自己认识这个哨兵值。

## 5. 分段实现：按顺序运行

**本章接口：** `smallest_sufficient_team(skills, people)`、`min_stickers(stickers, target)`、`hamiltonian_path_dp(cost)`

### 导入本章使用的库

In [1]:
from functools import cache
from collections import Counter

### smallest_sufficient_team

In [2]:
def smallest_sufficient_team(skills, people):
    pos = {s: i for i, s in enumerate(skills)}
    full = (1 << len(skills)) - 1
    dp = {0: ()}
    for i, person in enumerate(people):
        mask = 0
        for s in person:
            if s in pos:
                mask |= 1 << pos[s]
        for covered, team in list(dp.items()):
            new = covered | mask
            if new not in dp or len(team) + 1 < len(dp[new]):
                dp[new] = team + (i,)
    return list(dp[full]) if full in dp else None

### min_stickers

In [3]:
def min_stickers(stickers, target):
    letters = sorted(set(target))
    wanted = Counter(target)
    vectors = [tuple((Counter(s)[c] for c in letters)) for s in stickers]

    @cache
    def f(state):
        if not any(state):
            return 0
        first = next((i for i, x in enumerate(state) if x))
        best = float('inf')
        for v in vectors:
            if not v[first]:
                continue
            new = tuple((max(0, x - y) for x, y in zip(state, v)))
            best = min(best, 1 + f(new))
        return best
    answer = f(tuple((wanted[c] for c in letters)))
    return -1 if answer == float('inf') else answer

### hamiltonian_path_dp

In [4]:
def hamiltonian_path_dp(cost):
    n = len(cost)
    if not n:
        return 0
    if any((len(row) != n for row in cost)):
        raise ValueError('square cost matrix required')
    dp = [[float('inf')] * n for _ in range(1 << n)]
    for u in range(n):
        dp[1 << u][u] = 0
    for mask in range(1 << n):
        for u in range(n):
            if not mask >> u & 1:
                continue
            for v in range(n):
                if not mask >> v & 1:
                    dp[mask | 1 << v][v] = min(dp[mask | 1 << v][v], dp[mask][u] + cost[u][v])
    return min(dp[-1])

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

skills=['python','sql','js']; people=[['python'],['sql'],['sql','js']]
show_table(['人','技能','掩码'],[(i,p,format(sum(1<<skills.index(s) for s in set(p)),'03b')) for i,p in enumerate(people)])
print('最小团队',smallest_sufficient_team(skills,people))

人,技能,掩码
0,['python'],001
1,['sql'],010
2,"['sql', 'js']",110


最小团队 [0, 2]


## 7. 正确性、适用条件与复杂度

团队每人只处理一轮，读取快照防止本轮重复选择；对同一个覆盖mask，人数更多的方案被支配。贴纸选取顺序无关，可强制下一张覆盖首个未满足字符来剪枝；每次该坐标严格减少，递归终止。Hamiltonian状态按最后一条边追加一个未访问点，既无重复又覆盖所有路径。

**代价：** 团队O(P·2^S·S)保守时间（含元组复制），O(S·2^S)空间；贴纸状态数至多∏(字符需求数+1)；Hamiltonian O(n²2^n)时间O(n2^n)空间。缺失边用+∞，无可行Hamiltonian路径返回+∞。

### 展开理解

**为什么是对的？** 团队问题的论证：任何一个团队方案，把人按编号从小到大排好，让算法按"每人一轮"的顺序处理，它恰好会把这些人逐个并入，最终在"覆盖掩码"对应的键下登记这个方案（或更短的等价方案）。我们对每个掩码只保留最短的方案是安全的，因为后续扩展只看掩码不看人，短方案能做的长方案都能做且只会更差（被支配）。快照遍历保证一个人在一轮里最多被选一次，符合"每人最多进一次团队"的现实。贴纸问题的论证核心是"顺序无关"：无论最优解里贴纸按什么顺序使用，我们都可以把它重排成"先补首个缺口"的顺序而不改变用的张数（每张贴纸贡献的字母集合与使用时机无关，只与总共用了哪些张有关），所以强制这个顺序不丢最优解；而每步首个缺口坐标严格减小，保证了递归有限步内归零或判死。哈密顿的论证：任何一条访问顺序确定的路径，都唯一地分解为"一个点一个点往集合里加"的过程，每加一个点对应 DP 的一步转移，路径费用恰好逐段累加；反过来 DP 的每条转移链都对应一条真实路径。所以"全访问、停在各终点的最小 dp 值"就是"最短哈密顿路径"。状态设计上，mask 之外再带最后位置 u，正是"把影响未来的信息存全"——下一步的费用只依赖当前站在哪。

**复杂度**：团队问题每个掩码键的维护牵扯元组复制，保守估计 O(P·2^S·S) 时间（P 是人数、S 是技能数）、O(S·2^S) 空间；S=16 时 65536 个掩码，P=60 时也在千万级操作内，可跑。贴纸的状态数至多是"每种字母需求数 +1 的连乘"（每格从需求数降到 0），比朴素的指数界往往小得多。哈密顿是 O(n²·2ⁿ) 时间、O(n·2ⁿ) 空间：n=20 时约 4 亿步，偏慢但能跑；n=25 就到千亿级，基本不可行——这就是"压缩让它可控而非变成多项式"的体感。

## 8. 单元测试：每个用例在检查什么

- `assert smallest_sufficient_team(['a'],[[],['b']]) is None`：不可行测试——唯一需要的技能 'a' 没有人会，`full` 永远进不了 dp，必须返回 None 而不是崩溃。
- `assert smallest_sufficient_team([],[])==[]`：边界测试——没有技能要覆盖，`full=0`，dp[0]=() 就是答案，空团队；它专测初始化 `dp={0:()}` 和 `full` 的计算在 S=0 时 `(1<<0)-1=0` 是否自洽。
- `assert min_stickers(['with','example','science'],'thehat')==3`：正常值测试，第三节副线手算的 3 张（example、with、with）。
- `assert min_stickers(['notice','possible'],'basicbasic')==-1`：不可行测试——目标里的 'a' 两张贴纸一张都供不了，`f` 返回无穷，翻译成 −1。
- 后半部分是随机对拍：哈密顿部分对 n=1..7 各生成一张元素取 −2..7 的随机费用矩阵（**含负数**，所以不能贪心、也不能用 Dijkstra），裁判是 `permutations` 全排列暴力——枚举每个访问顺序，把相邻两城的费用加起来取最小。团队部分跑 100 轮：3 项技能、6 个人、每人随机持有子集；裁判先枚举 2⁶=64 个人选掩码，筛出技能并集恰好齐全的那些（`set().union(...)` 逐个并入成员技能）；若有可行方案，断言本章返回的团队人数等于可行方案的最小人数（`bit_count` 数 1 的个数）、团队技能并集确实覆盖全部技能、且没有重复选人；若无任何可行方案，断言返回 None。这三条分别对应"最优性、可行性、无重复"三个契约。

In [6]:
assert smallest_sufficient_team(['a'], [[], ['b']]) is None

assert smallest_sufficient_team([], []) == []

assert min_stickers(['with', 'example', 'science'], 'thehat') == 3

assert min_stickers(['notice', 'possible'], 'basicbasic') == -1

from itertools import permutations

from random import Random

rng = Random(112)

for n in range(1, 8):
    cost = [[rng.randrange(-2, 8) for _ in range(n)] for _ in range(n)]
    assert hamiltonian_path_dp(cost) == min((sum((cost[u][v] for u, v in zip(p, p[1:]))) for p in permutations(range(n))))

for _ in range(100):
    skills = ['a', 'b', 'c']
    people = [[s for s in skills if rng.randrange(2)] for _ in range(6)]
    team = smallest_sufficient_team(skills, people)
    valid = [mask for mask in range(1 << 6) if set().union(*(set(people[i]) for i in range(6) if mask >> i & 1)) == set(skills)]
    if valid:
        assert len(team) == min((x.bit_count() for x in valid))
        assert set().union(*(set(people[i]) for i in team)) == set(skills)
        assert len(team) == len(set(team))
    else:
        assert team is None

print('N112: 所有本章断言通过')

N112: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 为最小团队恢复人员索引。

**练习 2：** 区分O(2^n)状态和可能更多的转移。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（为最小团队恢复人员索引）**：提示——注意本章实现直接把元组当方案存，天然就带着索引。这个练习想让你做的是"假如你先写了一个只存人数的版子，怎么再把名单挖出来"。两条路：一是转移时多存一个"从哪个掩码、选了哪个人来的"前驱记录，从 `full` 倒着走回 0；二是利用“自动测试”部分裁判的思路反问自己——已知最优人数 k，能否从小到大枚举人的组合验证（那就是暴力了，仅用于对拍）。手算例子就用第一节的 3 人例子：前驱链是 000 →(选人0)→ 001 →(选人2)→ 111，反推出 [0,2]。边界别忘了 None 和空技能两种情形各自该返回什么。
- **练习 2（区分 O(2ⁿ) 状态和可能更多的转移）**：提示——给三个函数分别数两笔账。团队：状态是 2^S 个掩码，转移是"每个人 × 每个已有状态"共 P·2^S 次，再加元组复制的 S 因子。贴纸：状态是字母计数向量的取值组合（不是 2 的幂），转移是"每个状态 × 每张贴纸"。哈密顿：状态 2ⁿ·n 个，转移每个状态再乘 n（选下一个城市），总量 O(n²·2ⁿ)。用 n=20 亲手算一下：2²⁰ ≈ 10⁶ 个状态，每个若带 20 个终点就是 2×10⁷ 个格子，每个格子再试 20 条出边，就是 4×10⁸ 次转移——状态数百万、转移数上亿，量级差 20 倍以上，这就是"状态数 ≠ 转移数"的体感。手算演示建议拿 n=3 的费用表列出 8 个 mask × 3 个终点共 24 个格子，数一数其中多少被真实填充、多少次 min 比较发生。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from functools import cache

from collections import Counter

def smallest_sufficient_team(skills, people):
    pos = {s: i for i, s in enumerate(skills)}
    full = (1 << len(skills)) - 1
    dp = {0: ()}
    for i, person in enumerate(people):
        mask = 0
        for s in person:
            if s in pos:
                mask |= 1 << pos[s]
        for covered, team in list(dp.items()):
            new = covered | mask
            if new not in dp or len(team) + 1 < len(dp[new]):
                dp[new] = team + (i,)
    return list(dp[full]) if full in dp else None

def min_stickers(stickers, target):
    letters = sorted(set(target))
    wanted = Counter(target)
    vectors = [tuple((Counter(s)[c] for c in letters)) for s in stickers]

    @cache
    def f(state):
        if not any(state):
            return 0
        first = next((i for i, x in enumerate(state) if x))
        best = float('inf')
        for v in vectors:
            if not v[first]:
                continue
            new = tuple((max(0, x - y) for x, y in zip(state, v)))
            best = min(best, 1 + f(new))
        return best
    answer = f(tuple((wanted[c] for c in letters)))
    return -1 if answer == float('inf') else answer

def hamiltonian_path_dp(cost):
    n = len(cost)
    if not n:
        return 0
    if any((len(row) != n for row in cost)):
        raise ValueError('square cost matrix required')
    dp = [[float('inf')] * n for _ in range(1 << n)]
    for u in range(n):
        dp[1 << u][u] = 0
    for mask in range(1 << n):
        for u in range(n):
            if not mask >> u & 1:
                continue
            for v in range(n):
                if not mask >> v & 1:
                    dp[mask | 1 << v][v] = min(dp[mask | 1 << v][v], dp[mask][u] + cost[u][v])
    return min(dp[-1])

# 示例与回归测试
assert smallest_sufficient_team(['a'], [[], ['b']]) is None

assert smallest_sufficient_team([], []) == []

assert min_stickers(['with', 'example', 'science'], 'thehat') == 3

assert min_stickers(['notice', 'possible'], 'basicbasic') == -1

from itertools import permutations

from random import Random

rng = Random(112)

for n in range(1, 8):
    cost = [[rng.randrange(-2, 8) for _ in range(n)] for _ in range(n)]
    assert hamiltonian_path_dp(cost) == min((sum((cost[u][v] for u, v in zip(p, p[1:]))) for p in permutations(range(n))))

for _ in range(100):
    skills = ['a', 'b', 'c']
    people = [[s for s in skills if rng.randrange(2)] for _ in range(6)]
    team = smallest_sufficient_team(skills, people)
    valid = [mask for mask in range(1 << 6) if set().union(*(set(people[i]) for i in range(6) if mask >> i & 1)) == set(skills)]
    if valid:
        assert len(team) == min((x.bit_count() for x in valid))
        assert set().union(*(set(people[i]) for i in team)) == set(skills)
        assert len(team) == len(set(team))
    else:
        assert team is None

print('N112: 所有本章断言通过')

N112: 所有本章断言通过


## 11. 代表题与迁移

- **464. Can I Win**：练"把'哪些数字已被用'压成 mask，两人博弈在状态图上做记忆化搜索"，与本章 `min_stickers` 的 @cache + 状态元组是同款工具箱。
- **691. Stickers to Spell Word**：练"剩余需求向量当状态 + 强制覆盖首个缺口剪枝"，本章 `min_stickers` 即其解。
- **1125. Smallest Sufficient Team**：练"mask 当键、方案元组当值、支配保留最短、每人一轮快照遍历"，本章 `smallest_sufficient_team` 即其解。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。